**OBSOLETE -- superseded by the 2026-08-25 reorientation plan.**

Kept in the repo as a historical record (real, validated results at the
time -- e.g. the calibration submission that scored public LB 0.596) but
not maintained going forward. The gold+weak preprocessing/training/
inference pipeline built here is being rebuilt from scratch under the
new plan (measurement-gate fix, verified slice ordering, validated label
sets, a rebuilt `src/`-backed preprocessing pass, and a 6-slot
attention model), tracked in the `v2` notebooks
(`00v2_measurement_gate.ipynb`, `01v2_slice_ordering.ipynb`, ...). See
README.md for the current plan.

# 04b - Fase 5: mezclar gold+weak, GroupKFold por plantilla de informe

Corre en local contra `data/raw/` (solo texto/CSV, no hace falta Kaggle/GPU). Objetivo: construir la tabla combinada de los 4,407 estudios (58 gold con etiqueta oficial + 4,349 weak con etiqueta del labeler de la Fase 3) y asignar `fold` con un `GroupKFold` agrupado por `src/labelers.py::report_group_key()`, no por estudio, para no filtrar plantillas de informe compartidas entre train/val (incluido el caso confirmado en Fase 2 de una plantilla compartida entre un estudio gold y uno weak).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.labelers import report_group_key, label_reports
from src.config import FINDINGS, OFFICIAL_LABEL_COLUMNS

RAW_DIR = REPO_ROOT / "data" / "raw"
train = pd.read_csv(RAW_DIR / "train.csv")
label_cols = list(OFFICIAL_LABEL_COLUMNS.values())

n_present = train[label_cols].notna().sum(axis=1)
is_gold = n_present == len(label_cols)
print(f"Total estudios: {len(train)}")
print(f"Gold: {is_gold.sum()}, weak: {(~is_gold).sum()}")
assert is_gold.sum() == 58

Total estudios: 4407
Gold: 58, weak: 4349


## Plantillas de informe (report_group_key) y validacion contra la Fase 2

report_group_key() ya esta implementado en src/labelers.py (2026-08-18). Validacion: debe reproducir exactamente los 54 grupos duplicados / 206 estudios / 1 mezcla gold-weak medidos en notebooks/02_eda_reports.ipynb seccion E.

In [2]:
train["group_key"] = train["Report"].apply(report_group_key)
group_sizes = train.groupby("group_key").size()
dup_groups = group_sizes[group_sizes > 1]
print(f"Grupos duplicados: {len(dup_groups)}")
print(f"Estudios en grupo duplicado: {dup_groups.sum()}")

train["is_gold"] = is_gold.values
mixed = train.groupby("group_key")["is_gold"].agg(["any", "all"])
mixed_groups = mixed[(mixed["any"]) & (~mixed["all"])]
print(f"Grupos con mezcla gold+weak: {len(mixed_groups)}")

assert len(dup_groups) == 54, f"esperados 54 grupos, hay {len(dup_groups)}"
assert dup_groups.sum() == 206, f"esperados 206 estudios, hay {dup_groups.sum()}"
assert len(mixed_groups) == 1, f"esperada 1 mezcla gold-weak, hay {len(mixed_groups)}"
print("OK: coincide exactamente con la Fase 2.")

Grupos duplicados: 54
Estudios en grupo duplicado: 206
Grupos con mezcla gold+weak: 1
OK: coincide exactamente con la Fase 2.


## GroupKFold sobre los 4,407 estudios, agrupado por plantilla

CV_FOLDS=5 y RANDOM_STATE=42 (src/config.py) tenian valor puesto pero no se usaban en ningun sitio del repo hasta ahora (confirmado en la auditoria del 2026-08-18). sklearn.GroupKFold no soporta shuffle/random_state (asigna folds de forma deterministica por como aparecen los grupos), asi que RANDOM_STATE no aplica aqui directamente -- se deja documentado por si la Fase 6 lo necesita para otra cosa (inicializacion de modelo, etc).

In [3]:
from sklearn.model_selection import GroupKFold
from src.config import CV_FOLDS

gkf = GroupKFold(n_splits=CV_FOLDS)
train["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(gkf.split(train, groups=train["group_key"])):
    train.loc[train.index[val_idx], "fold"] = fold_idx

assert (train["fold"] >= 0).all()
print("Estudios por fold:")
print(train["fold"].value_counts().sort_index())

# Verificar que ningun grupo (plantilla de informe) se reparte entre folds
folds_per_group = train.groupby("group_key")["fold"].nunique()
leaking_groups = folds_per_group[folds_per_group > 1]
print(f"Grupos repartidos entre folds (deberia ser 0): {len(leaking_groups)}")
assert len(leaking_groups) == 0

# Gold por fold
print("Estudios gold por fold:")
print(train.loc[train["is_gold"], "fold"].value_counts().sort_index())

Estudios por fold:
fold
0    882
1    882
2    881
3    881
4    881
Name: count, dtype: int64
Grupos repartidos entre folds (deberia ser 0): 0
Estudios gold por fold:
fold
0    16
1    11
2    14
3     8
4     9
Name: count, dtype: int64


## Tabla de entrenamiento combinada: gold (etiqueta oficial) + weak (labeler)

Gold usa las 12 columnas oficiales tal cual (0/1 exactos, nunca se pasan por el labeler). Weak usa `label_reports()` de la Fase 3 (ya con el fix de negacion del 2026-08-18): valores en {0.0, 0.5, 1.0}, donde 0.5 es abstencion. Se guarda tambien `is_gold` para que el entrenamiento pueda tratar las dos fuentes distinto si hace falta (p.ej. mas peso a gold, o excluir 0.5 de la loss).

In [4]:
weak_mask = ~train["is_gold"]

weak_labels = label_reports(train.loc[weak_mask, ["StudyInstanceUID", "Report"]], FINDINGS)
weak_labels = weak_labels.reindex(train.loc[weak_mask, "StudyInstanceUID"]).reset_index(drop=True)

gold_labels = train.loc[train["is_gold"], label_cols].reset_index(drop=True)
gold_labels.columns = FINDINGS

combined = pd.concat([
    train.loc[train["is_gold"], ["StudyInstanceUID", "fold", "is_gold"]].reset_index(drop=True).join(gold_labels),
    train.loc[weak_mask, ["StudyInstanceUID", "fold", "is_gold"]].reset_index(drop=True).join(weak_labels),
], ignore_index=True)

print(f"Filas combinadas: {len(combined)} (esperado 4407)")
assert len(combined) == 4407
assert combined["StudyInstanceUID"].nunique() == 4407

# Gold en la tabla combinada debe ser identico a las columnas oficiales, sin pasar por el labeler
check = combined[combined["is_gold"]].merge(
    train.loc[train["is_gold"], ["StudyInstanceUID"] + label_cols], on="StudyInstanceUID",
)
mismatches = 0
for finding, col in zip(FINDINGS, label_cols):
    mismatches += (check[finding] != check[col]).sum()
print(f"Discrepancias gold combinado vs columnas oficiales (debe ser 0): {mismatches}")
assert mismatches == 0

print("Distribucion de valores weak por hallazgo (0.0 / 0.5 / 1.0):")
weak_only = combined[~combined["is_gold"]]
print(weak_only[FINDINGS].apply(lambda s: s.value_counts(normalize=True).round(3)).T)

Filas combinadas: 4407 (esperado 4407)
Discrepancias gold combinado vs columnas oficiales (debe ser 0): 0
Distribucion de valores weak por hallazgo (0.0 / 0.5 / 1.0):
                                 0.0    0.5    1.0
acl_injury                     0.171  0.747  0.082
mcl_injury                     0.182  0.765  0.052
medial_meniscus_tear           0.124  0.702  0.173
lateral_meniscus_tear          0.207  0.714  0.079
oa_medial_compartment          0.072  0.851  0.077
oa_lateral_compartment         0.082  0.863  0.054
oa_patellofemoral_compartment  0.105  0.793  0.102
effusion                       0.165  0.542  0.293
synovitis                      0.004  0.894  0.101
bakers_cyst                    0.110  0.765  0.125
bone_contusion                 0.047  0.869  0.084
fracture                       0.114  0.840  0.046
